In [8]:
import os

os.environ['HTTP_PROXY'] = '127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = '127.0.0.1:7890'
import json

from urllib.request import urlopen
from PIL import Image
import torch
from huggingface_hub import hf_hub_download
from open_clip import create_model_and_transforms, get_tokenizer
from open_clip.factory import HF_HUB_PREFIX, _MODEL_CONFIGS


# Download the model and config files
hf_hub_download(
    repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    filename="open_clip_pytorch_model.bin",
    local_dir="checkpoints"
)
hf_hub_download(
    repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    filename="open_clip_config.json",
    local_dir="checkpoints"
)


# Load the model and config files
model_name = "biomedclip_local"

with open("checkpoints/open_clip_config.json", "r") as f:
    config = json.load(f)
    model_cfg = config["model_cfg"]
    preprocess_cfg = config["preprocess_cfg"]


if (not model_name.startswith(HF_HUB_PREFIX)
    and model_name not in _MODEL_CONFIGS
    and config is not None):
    _MODEL_CONFIGS[model_name] = model_cfg

tokenizer = get_tokenizer(model_name)

model, _, preprocess = create_model_and_transforms(
    model_name=model_name,
    pretrained="checkpoints/open_clip_pytorch_model.bin",
    **{f"image_{k}": v for k, v in preprocess_cfg.items()},
)


In [9]:
from torchvision import transforms
import random
from collections import defaultdict
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
from PIL import Image
import torch

class FewShotImageFolder(Dataset):
    def __init__(self, root, num_shots, transform=None, test_transform=None, seed=42):
        super().__init__()
        self.root = root
        self.num_shots = num_shots
        self.transform = transform
        self.test_transform = test_transform
        
        full_dataset = ImageFolder(root)
        self.class_to_idx = full_dataset.class_to_idx

        label_to_samples = defaultdict(list)
        for path, label in full_dataset.samples:
            label_to_samples[label].append(path)
        
        random.seed(seed)
        
        self.train_samples = []
        self.train_targets = []
        self.test_samples = []
        self.test_targets = []

        for label, samples in label_to_samples.items():
            if len(samples) <= num_shots:
                sampled = samples
                remain = []
            else:
                sampled = random.sample(samples, num_shots)
                remain = [s for s in samples if s not in sampled]

            self.train_samples.extend(sampled)
            self.train_targets.extend([label] * len(sampled))
            self.test_samples.extend(remain)
            self.test_targets.extend([label] * len(remain))

        # 默认使用训练集
        combined = list(zip(self.train_samples, self.train_targets))
        random.shuffle(combined)
        self.samples, self.targets = zip(*combined)
        self.samples = list(self.samples)
        self.targets = list(self.targets)
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.targets[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label)
        return image, label

    def get_test_set(self):
        return SimpleImageDataset(self.test_samples, self.test_targets,self.test_transform)


class SimpleImageDataset(Dataset):
    def __init__(self, samples, targets, transform):
        self.samples = samples
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.targets[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label)
        return image, label


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
import numpy as np

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_):
    return GradReverse.apply(x, lambda_)

class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1),
            # nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

class GatedAdapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = dim * 2
        self.adapter = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )
        # self.gate = nn.Parameter(torch.tensor(1.0))  # learnable weight

    def forward(self, x):
        # return x + self.gate * self.adapter(x)
        return x + self.adapter(x)


class CLIP_DA(nn.Module): ## LD: Logits Deconfusion

    def __init__(self, embed_dim, clip_model):
        super().__init__()
     
        feature_dim = embed_dim  # 1024 / 512
        self.clip_model = clip_model
  
        self.image_adapter = GatedAdapter(feature_dim)
        self.text_adapter = GatedAdapter(feature_dim)
        # self.image_adapter = PromptAwareMoEAdapter(feature_dim)
        # self.text_adapter = PromptAwareMoEAdapter(feature_dim)

        # self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.logit_scale = nn.Parameter(torch.tensor(np.log(clip_model.logit_scale.exp().cpu().detach().numpy()))) 
        # self.logit_scale = nn.Parameter(torch.tensor(np.log(50)))
        # self.logit_scale.requires_grad = False


        self.initialize_parameters()

    def initialize_parameters(self):

        for param in self.clip_model.parameters():
            param.requires_grad = False

    def forward_OriCLIP(self, image, text):
        image_features = self.encode_image(image)
        text_features = self.encode_text(text)

        # normalized features
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # cosine similarity as logits
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()

        return logits_per_image, logits_per_text
    
    def encode_text(self,text_tokens):
        text_feats = self.clip_model.encode_text(text_tokens)
        # text_feats = text_feats + self.adapter["adapter_2"](text_feats) ## If adapt text features
        return text_feats

    def forward(self, images, text_feats):  ## Few-shot ensemble text prompts
  
        image_feats = self.clip_model.encode_image(images)
        image_feats = self.image_adapter(image_feats)
        image_logits = F.normalize(image_feats, dim=-1)

        # text_feats = self.clip_model.encode_text(text) ## cause we need to ensemble prompts before
        text_feats =  self.text_adapter(text_feats) ## adapt text features option
        text_logits = F.normalize(text_feats, dim=-1)

        return image_logits,text_logits

def contrastive_loss(img_feats, txt_feats, logit_scale):
    logits_per_image = logit_scale.exp() * (img_feats @ txt_feats.T)  # [B, dim]
    logits_per_text = logits_per_image.T  # [B, dim]

    labels = torch.arange(img_feats.size(0)).to(img_feats.device)  

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2

## AAM

In [ ]:
## 2026
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score


# =========================================================
# 1. Feature extraction
# =========================================================
def extract_image_features(model, images, layers):
    features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            features[idx] = output[:, 0]
        return hook

    trunk = model.visual.trunk
    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    return [features[idx] for idx in layers]


def extract_image_features_withproj(model, images, layers=[5, 7, 9, 11]):
    features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            features[idx] = output[:, 0]
        return hook

    trunk = model.visual.trunk
    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    feats_out = []
    for idx in layers:
        proj = model.visual.head(features[idx])
        feats_out.append(F.normalize(proj, dim=-1))

    return feats_out


# =========================================================
# 2. Dynamic layer scoring / selection / weighting
# =========================================================
def compute_cls_layer_scores(model, train_loader, candidate_layers, device):
    layer_scores = {}

    with torch.no_grad():
        feats_per_layer = {l: defaultdict(list) for l in candidate_layers}

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            layer_feats = extract_image_features(model, images, candidate_layers)

            for i, l in enumerate(candidate_layers):
                feats = F.normalize(layer_feats[i], dim=-1)
                for j, y in enumerate(labels):
                    feats_per_layer[l][y.item()].append(feats[j])

        for l in candidate_layers:
            class_feats = feats_per_layer[l]
            class_ids = sorted(class_feats.keys())

            inter_sims = []
            for i in range(len(class_ids)):
                for j in range(i + 1, len(class_ids)):
                    proto_i = torch.stack(class_feats[class_ids[i]]).mean(0)
                    proto_j = torch.stack(class_feats[class_ids[j]]).mean(0)
                    sim = F.cosine_similarity(proto_i, proto_j, dim=0).item()
                    inter_sims.append(sim)

            layer_scores[l] = 1.0 - np.mean(inter_sims) if len(inter_sims) > 0 else 0.0

    return layer_scores


def select_best_layers(layer_scores, top_k=4, final_layer=None):
    items = list(layer_scores.items())

    if final_layer is not None and final_layer in layer_scores and top_k >= 1:
        items_wo_final = [(l, s) for l, s in items if l != final_layer]
        items_wo_final = sorted(items_wo_final, key=lambda x: x[1], reverse=True)

        selected = [l for l, _ in items_wo_final[:max(top_k - 1, 0)]]
        selected.append(final_layer)
    else:
        items = sorted(items, key=lambda x: x[1], reverse=True)
        selected = [l for l, _ in items[:top_k]]

    return sorted(list(set(selected)))


def compute_adaptive_layer_weights(layer_scores, selected_layers, tau=1.0, mode="softmax"):
    scores = np.array(
        [layer_scores.get(l, 1e-6) for l in selected_layers],
        dtype=np.float64
    )

    if mode == "softmax":
        scores = scores / max(tau, 1e-6)
        scores = scores - scores.max()
        weights = np.exp(scores)
        weights = weights / weights.sum()
    elif mode == "linear":
        scores = np.maximum(scores, 1e-12)
        weights = scores / scores.sum()
    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return weights.tolist()


def prepare_dynamic_layer_config(
    model,
    train_loader,
    device,
    candidate_middle_layers=[2, 5, 8, 11],
    topk_middle=4,
    final_middle_layer=11,
    tau_middle=1.0,
    weight_mode="softmax",
):
    middle_scores = compute_cls_layer_scores(
        model, train_loader, candidate_middle_layers, device
    )

    selected_middle_layers = select_best_layers(
        middle_scores,
        top_k=topk_middle,
        final_layer=final_middle_layer
    )

    middle_weights = compute_adaptive_layer_weights(
        middle_scores,
        selected_middle_layers,
        tau=tau_middle,
        mode=weight_mode
    )

    return {
        "middle_scores": middle_scores,
        "selected_middle_layers": selected_middle_layers,
        "middle_weights": middle_weights,
    }


# =========================================================
# 3. Model blocks
# =========================================================
class GatedAdapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = dim * 2
        self.adapter = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return x + self.adapter(x)


class CLIP_DA_GlobalMiddle(nn.Module):
    """
    Global + middle-level image-text alignment adapter.

    Important:
    - No learnable classifier head.
    - Final fusion is trained on support-set image-text similarity logits.
    """
    def __init__(
        self,
        embed_dim,
        clip_model,
        middle_layers=[2, 5, 8, 11],
        use_middle=True,
    ):
        super().__init__()

        self.clip_model = clip_model
        self.embed_dim = embed_dim
        self.middle_layers = middle_layers
        self.use_middle = use_middle

        self.image_adapter = GatedAdapter(embed_dim)
        self.text_adapter = GatedAdapter(embed_dim)
        self.middle_adapter = GatedAdapter(embed_dim)

        self.logit_scale = nn.Parameter(
            torch.tensor(
                np.log(clip_model.logit_scale.exp().detach().cpu().item()),
                dtype=torch.float32
            )
        )

        # Learnable fusion coefficient trained on support set.
        # softplus(log_beta_middle) ensures beta > 0.
        self.log_beta_middle = nn.Parameter(torch.tensor(0.0))

        self.initialize_parameters()

    def initialize_parameters(self):
        for param in self.clip_model.parameters():
            param.requires_grad = False

    def beta_middle(self):
        return F.softplus(self.log_beta_middle)

    def encode_text(self, text_tokens):
        return self.clip_model.encode_text(text_tokens)

    def adapt_text_features(self, text_feats):
        text_feats = self.text_adapter(text_feats)
        return F.normalize(text_feats, dim=-1)

    def forward_global(self, images):
        global_img_feats = self.clip_model.encode_image(images)
        global_img_feats = self.image_adapter(global_img_feats)
        return F.normalize(global_img_feats, dim=-1)

    def forward_middle(self, images):
        if not self.use_middle:
            return None

        middle_list = extract_image_features_withproj(
            self.clip_model,
            images,
            self.middle_layers
        )

        middle_feats = torch.stack(middle_list, dim=1)
        middle_feats = self.middle_adapter(middle_feats)
        return F.normalize(middle_feats, dim=-1)

    def forward(self, images, text_feats):
        global_img_feats = self.forward_global(images)
        adapted_text_feats = self.adapt_text_features(text_feats)
        middle_feats = self.forward_middle(images)

        return global_img_feats, adapted_text_feats, middle_feats


# =========================================================
# 4. Losses and aggregations
# =========================================================
def contrastive_loss(img_feats, txt_feats, logit_scale):
    logits_per_image = logit_scale.exp() * (img_feats @ txt_feats.T)
    logits_per_text = logits_per_image.T

    labels = torch.arange(img_feats.size(0), device=img_feats.device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)

    return (loss_i + loss_t) / 2


def aggregate_middle_logits(
    raw_middle_logits,
    layer_weights=None,
    pooling="weighted"
):
    B, Lm, C = raw_middle_logits.shape

    if pooling == "mean":
        return raw_middle_logits.mean(dim=1)

    elif pooling == "max":
        return raw_middle_logits.max(dim=1).values

    elif pooling == "weighted":
        if layer_weights is None:
            weights = torch.ones(
                Lm,
                device=raw_middle_logits.device,
                dtype=raw_middle_logits.dtype
            ) / Lm
        else:
            weights = torch.tensor(
                layer_weights,
                device=raw_middle_logits.device,
                dtype=raw_middle_logits.dtype
            )
            weights = weights / weights.sum()

        return (raw_middle_logits * weights.view(1, Lm, 1)).sum(dim=1)

    else:
        raise ValueError(f"Unsupported pooling: {pooling}")


def middle_alignment_loss(
    middle_feats,
    class_text_feats,
    labels,
    logit_scale,
    layer_weights=None,
    pooling="weighted"
):
    raw_logits = logit_scale.exp() * torch.einsum(
        "bld,cd->blc",
        middle_feats,
        class_text_feats
    )

    image_class_logits = aggregate_middle_logits(
        raw_logits,
        layer_weights=layer_weights,
        pooling=pooling
    )

    loss = F.cross_entropy(image_class_logits, labels)

    return loss, image_class_logits, raw_logits


# =========================================================
# 5. Text feature builders
# =========================================================
@torch.no_grad()
def build_class_text_features(
    clip_model,
    tokenizer,
    class_names,
    prompt_templates,
    device
):
    prompts = [
        template.format(class_name)
        for class_name in class_names
        for template in prompt_templates
    ]

    tokens = tokenizer(prompts).to(device)

    text_features = clip_model.encode_text(tokens)
    text_features = text_features.view(
        len(class_names),
        len(prompt_templates),
        -1
    )
    text_features = text_features.mean(dim=1)
    text_features = F.normalize(text_features, dim=-1)

    return text_features


# =========================================================
# 6. Training
# =========================================================
def train_global_middle_dynamic(
    model,
    tokenizer,
    train_loader,
    class_names,
    prompt_templates,
    device,
    embed_dim=512,
    candidate_middle_layers=[2, 5, 8, 11],
    topk_middle=4,
    final_middle_layer=11,
    tau_middle=1.0,
    layer_weight_mode="softmax",
    use_middle=True,
    lambda_global=0.1,
    lambda_middle=0.1,
    lr=1e-3,
    num_epochs=30,
    middle_pooling="weighted",
):
    layer_config = prepare_dynamic_layer_config(
        model=model,
        train_loader=train_loader,
        device=device,
        candidate_middle_layers=candidate_middle_layers,
        topk_middle=topk_middle,
        final_middle_layer=final_middle_layer,
        tau_middle=tau_middle,
        weight_mode=layer_weight_mode,
    )

    selected_middle_layers = layer_config["selected_middle_layers"]
    middle_weights = layer_config["middle_weights"]

    print("Selected middle layers:", selected_middle_layers)
    print("Middle layer weights:", [round(w, 4) for w in middle_weights])

    adapter_model = CLIP_DA_GlobalMiddle(
        embed_dim=embed_dim,
        clip_model=model,
        middle_layers=selected_middle_layers,
        use_middle=use_middle,
    ).to(device)

    optimizer_model = torch.optim.AdamW(
        adapter_model.parameters(),
        lr=lr,
        weight_decay=1e-4,
        eps=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_model,
        T_max=num_epochs * len(train_loader)
    )

    with torch.no_grad():
        cached_class_text_feats = build_class_text_features(
            clip_model=model,
            tokenizer=tokenizer,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device
        )

    best_state = None
    best_train_total = float("inf")

    for epoch in range(num_epochs):
        adapter_model.train()

        running_global_loss = 0.0
        running_middle_loss = 0.0
        running_fused_loss = 0.0
        running_total_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.size(0)

            # -------------------------------------------------
            # Instance-level text features for global contrastive alignment
            # -------------------------------------------------
            prompts_instance = [
                template.format(class_names[l.item()])
                for l in labels
                for template in prompt_templates
            ]

            tokens_instance = tokenizer(prompts_instance).to(device)

            with torch.no_grad():
                text_feats_instance = model.encode_text(tokens_instance)
                text_feats_instance = text_feats_instance.view(
                    batch_size,
                    len(prompt_templates),
                    -1
                )
                text_feats_instance = text_feats_instance.mean(dim=1)
                text_feats_instance = F.normalize(text_feats_instance, dim=-1)

            global_img_feats, adapted_instance_txt_feats, middle_feats = adapter_model(
                images,
                text_feats_instance
            )

            # -------------------------------------------------
            # Global instance-level image-text contrastive loss
            # -------------------------------------------------
            loss_global = contrastive_loss(
                global_img_feats,
                adapted_instance_txt_feats,
                adapter_model.logit_scale
            )

            # -------------------------------------------------
            # Class-wise text prototypes
            # -------------------------------------------------
            adapted_class_txt_feats = adapter_model.adapt_text_features(
                cached_class_text_feats
            )

            scale = adapter_model.logit_scale.exp()

            # -------------------------------------------------
            # Global class-wise image-text similarity logits
            # -------------------------------------------------
            global_logits = scale * (global_img_feats @ adapted_class_txt_feats.T)

            # -------------------------------------------------
            # Middle class-wise image-text similarity logits
            # -------------------------------------------------
            if use_middle and middle_feats is not None:
                loss_middle, middle_logits, _ = middle_alignment_loss(
                    middle_feats=middle_feats,
                    class_text_feats=adapted_class_txt_feats,
                    labels=labels,
                    logit_scale=adapter_model.logit_scale,
                    layer_weights=middle_weights,
                    pooling=middle_pooling
                )
            else:
                loss_middle = torch.tensor(0.0, device=device)
                middle_logits = torch.zeros_like(global_logits)

            # -------------------------------------------------
            # Support-set trained fusion of image-text similarity logits
            # -------------------------------------------------
            beta_middle = adapter_model.beta_middle()
            fused_logits = global_logits + beta_middle * middle_logits

            # This is still alignment-level CE:
            # logits are image-text similarities, not classifier outputs.
            loss_fused = F.cross_entropy(fused_logits, labels)

            total_loss = (
                loss_fused
                + lambda_global * loss_global
                + lambda_middle * loss_middle
            )

            optimizer_model.zero_grad()
            total_loss.backward()
            optimizer_model.step()
            scheduler.step()

            running_global_loss += loss_global.item() * batch_size
            running_middle_loss += loss_middle.item() * batch_size
            running_fused_loss += loss_fused.item() * batch_size
            running_total_loss += total_loss.item() * batch_size
            total_samples += batch_size

        avg_global = running_global_loss / total_samples
        avg_middle = running_middle_loss / total_samples
        avg_fused = running_fused_loss / total_samples
        avg_total = running_total_loss / total_samples

        if avg_total < best_train_total:
            best_train_total = avg_total
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in adapter_model.state_dict().items()
            }

        print(
            f"Epoch {epoch+1:02d} | "
            f"global_loss: {avg_global:.4f} | "
            f"middle_loss: {avg_middle:.4f} | "
            f"fused_loss: {avg_fused:.4f} | "
            f"total_loss: {avg_total:.4f} | "
            f"logit_scale: {adapter_model.logit_scale.exp().item():.4f} | "
            f"beta_middle: {adapter_model.beta_middle().item():.4f}"
        )

    if best_state is not None:
        adapter_model.load_state_dict(best_state)

    train_meta = {
        "prompt_templates": prompt_templates,
        "layer_config": layer_config,
        "beta_middle": adapter_model.beta_middle().detach().cpu().item(),
    }

    return adapter_model, train_meta


# =========================================================
# 7. Evaluation
# =========================================================
@torch.no_grad()
def evaluate_global_middle_dynamic(
    adapter_model,
    clip_model,
    data_loader,
    tokenizer,
    class_names,
    prompt_templates,
    device,
    middle_weights=None,
    use_middle=True,
    max_batch=None,
    middle_pooling="weighted",
    return_details=False
):
    adapter_model.eval()

    class_text_feats = build_class_text_features(
        clip_model=clip_model,
        tokenizer=tokenizer,
        class_names=class_names,
        prompt_templates=prompt_templates,
        device=device
    )

    all_labels = []
    all_global_logits = []
    all_middle_logits = []
    all_fused_logits = []
    all_probs = []
    all_preds = []

    total_loss = 0.0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(data_loader):
        if max_batch is not None and batch_idx >= max_batch:
            break

        images = images.to(device)
        labels = labels.to(device)
        batch_size = images.size(0)

        global_img_feats, adapted_class_txt_feats, middle_feats = adapter_model(
            images,
            class_text_feats
        )

        scale = adapter_model.logit_scale.exp()
        global_logits = scale * (global_img_feats @ adapted_class_txt_feats.T)

        if use_middle and middle_feats is not None:
            _, middle_logits, _ = middle_alignment_loss(
                middle_feats=middle_feats,
                class_text_feats=adapted_class_txt_feats,
                labels=labels,
                logit_scale=adapter_model.logit_scale,
                layer_weights=middle_weights,
                pooling=middle_pooling
            )
        else:
            middle_logits = torch.zeros_like(global_logits)

        beta_middle = adapter_model.beta_middle()
        fused_logits = global_logits + beta_middle * middle_logits

        probs = F.softmax(fused_logits, dim=1)
        preds = probs.argmax(dim=1)
        loss = F.cross_entropy(fused_logits, labels)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.append(labels.cpu())
        all_global_logits.append(global_logits.cpu())
        all_middle_logits.append(middle_logits.cpu())
        all_fused_logits.append(fused_logits.cpu())
        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_global_logits = torch.cat(all_global_logits).numpy()
    all_middle_logits = torch.cat(all_middle_logits).numpy()
    all_fused_logits = torch.cat(all_fused_logits).numpy()
    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()

    results = {
        "loss": total_loss / max(total_samples, 1),
        "acc": (all_preds == all_labels).mean(),
        "beta_middle": adapter_model.beta_middle().detach().cpu().item(),
    }

    if return_details:
        results.update({
            "labels": all_labels,
            "global_logits": all_global_logits,
            "middle_logits": all_middle_logits,
            "fused_logits": all_fused_logits,
            "probs": all_probs,
            "preds": all_preds,
        })

    return results

In [ ]:
## 2026
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score


# =========================================================
# 1. Feature extraction
# =========================================================
def extract_image_features(model, images, layers):
    """
    Extract intermediate CLS features without projection.
    Returns:
        list of tensors, each [B, C]
    """
    features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            features[idx] = output[:, 0]  # CLS token
        return hook

    trunk = model.visual.trunk
    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    return [features[idx] for idx in layers]


def extract_image_features_withproj(model, images, layers=[5, 7, 9, 11]):
    """
    Extract intermediate CLS features with projection.
    Returns:
        list of tensors, each [B, D]
    """
    features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            features[idx] = output[:, 0]  # CLS token
        return hook

    trunk = model.visual.trunk
    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    feats_out = []
    for idx in layers:
        proj = model.visual.head(features[idx])   # [B, D]
        feats_out.append(F.normalize(proj, dim=-1))
    return feats_out


# =========================================================
# 2. Dynamic layer scoring / selection / weighting
# =========================================================
def compute_cls_layer_scores(model, train_loader, candidate_layers, device):
    """
    Score candidate CLS layers by inter-class separability.
    Higher score = better.
    """
    layer_scores = {}

    with torch.no_grad():
        feats_per_layer = {l: defaultdict(list) for l in candidate_layers}

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            layer_feats = extract_image_features(model, images, candidate_layers)

            for i, l in enumerate(candidate_layers):
                feats = F.normalize(layer_feats[i], dim=-1)
                for j, y in enumerate(labels):
                    feats_per_layer[l][y.item()].append(feats[j])

        for l in candidate_layers:
            class_feats = feats_per_layer[l]
            class_ids = sorted(class_feats.keys())

            inter_sims = []
            for i in range(len(class_ids)):
                for j in range(i + 1, len(class_ids)):
                    proto_i = torch.stack(class_feats[class_ids[i]]).mean(0)
                    proto_j = torch.stack(class_feats[class_ids[j]]).mean(0)
                    inter_sims.append(F.cosine_similarity(proto_i, proto_j, dim=0).item())

            layer_scores[l] = 1.0 - np.mean(inter_sims) if len(inter_sims) > 0 else 0.0

    return layer_scores


def select_best_layers(layer_scores, top_k=4, final_layer=None):
    """
    Select top-k layers, optionally forcing final_layer to be included.
    """
    items = list(layer_scores.items())

    if final_layer is not None and final_layer in layer_scores and top_k >= 1:
        items_wo_final = [(l, s) for l, s in items if l != final_layer]
        items_wo_final = sorted(items_wo_final, key=lambda x: x[1], reverse=True)

        selected = [l for l, _ in items_wo_final[:max(top_k - 1, 0)]]
        if final_layer not in selected:
            selected.append(final_layer)
    else:
        items = sorted(items, key=lambda x: x[1], reverse=True)
        selected = [l for l, _ in items[:top_k]]

    selected = sorted(list(set(selected)))
    return selected


def compute_adaptive_layer_weights(layer_scores, selected_layers, tau=1.0, mode="softmax"):
    """
    Compute adaptive weights for selected layers.

    mode:
        - "softmax": temperature-scaled softmax
        - "linear": normalized linear scores
    """
    scores = np.array([layer_scores.get(l, 1e-6) for l in selected_layers], dtype=np.float64)

    if mode == "softmax":
        scores = scores / max(tau, 1e-6)
        scores = scores - scores.max()
        weights = np.exp(scores)
        weights = weights / weights.sum()
    elif mode == "linear":
        scores = tau * scores
        weights = scores / max(scores.sum(), 1e-12)
    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return weights.tolist()


def prepare_dynamic_layer_config(
    model,
    train_loader,
    device,
    candidate_middle_layers=[2, 5, 8, 11],
    topk_middle=4,
    final_middle_layer=11,
    tau_middle=1.0,
    weight_mode="softmax",
):
    """
    Compute dynamic layer selection + weights for middle branch only.
    """
    middle_scores = compute_cls_layer_scores(model, train_loader, candidate_middle_layers, device)

    selected_middle_layers = select_best_layers(
        middle_scores, top_k=topk_middle, final_layer=final_middle_layer
    )

    middle_weights = compute_adaptive_layer_weights(
        middle_scores, selected_middle_layers, tau=tau_middle, mode=weight_mode
    )

    layer_config = {
        "middle_scores": middle_scores,
        "selected_middle_layers": selected_middle_layers,
        "middle_weights": middle_weights,
    }
    return layer_config


# =========================================================
# 3. Model blocks
# =========================================================
class GatedAdapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = dim * 2
        self.adapter = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return x + self.adapter(x)


class CLIP_DA_GlobalMiddle(nn.Module):
    """
    Global alignment + optional middle-level CLS alignment.
    Dynamic layer selection and weighting are supported externally.
    """
    def __init__(
        self,
        embed_dim,
        clip_model,
        middle_layers=[2, 5, 8, 11],
        use_middle=True,
    ):
        super().__init__()

        self.clip_model = clip_model
        self.embed_dim = embed_dim
        self.middle_layers = middle_layers
        self.use_middle = use_middle

        self.image_adapter = GatedAdapter(embed_dim)
        self.text_adapter = GatedAdapter(embed_dim)
        self.middle_adapter = GatedAdapter(embed_dim)

        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(clip_model.logit_scale.exp().cpu().detach().numpy()))
        )

        self.initialize_parameters()

    def initialize_parameters(self):
        for param in self.clip_model.parameters():
            param.requires_grad = False

    def encode_text(self, text_tokens):
        return self.clip_model.encode_text(text_tokens)

    def adapt_text_features(self, text_feats):
        text_feats = self.text_adapter(text_feats)
        text_feats = F.normalize(text_feats, dim=-1)
        return text_feats

    def forward_global(self, images):
        global_img_feats = self.clip_model.encode_image(images)   # [B, D]
        global_img_feats = self.image_adapter(global_img_feats)
        global_img_feats = F.normalize(global_img_feats, dim=-1)
        return global_img_feats

    def forward_middle(self, images):
        if not self.use_middle:
            return None

        middle_list = extract_image_features_withproj(
            self.clip_model, images, self.middle_layers
        )   # list of [B, D]
        middle_feats = torch.stack(middle_list, dim=1)   # [B, Lm, D]
        middle_feats = self.middle_adapter(middle_feats)
        middle_feats = F.normalize(middle_feats, dim=-1)
        return middle_feats

    def forward(self, images, text_feats):
        global_img_feats = self.forward_global(images)
        adapted_text_feats = self.adapt_text_features(text_feats)
        middle_feats = self.forward_middle(images)

        return global_img_feats, adapted_text_feats, middle_feats


# =========================================================
# 4. Losses and aggregations
# =========================================================
def contrastive_loss(img_feats, txt_feats, logit_scale):
    logits_per_image = logit_scale.exp() * (img_feats @ txt_feats.T)
    logits_per_text = logits_per_image.T

    labels = torch.arange(img_feats.size(0), device=img_feats.device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2


def aggregate_middle_logits(
    raw_middle_logits,      # [B, Lm, C]
    layer_weights=None,
    pooling="weighted"
):
    """
    Aggregate middle-layer logits.
    """
    B, Lm, C = raw_middle_logits.shape

    if pooling == "mean":
        middle_logits = raw_middle_logits.mean(dim=1)
    elif pooling == "max":
        middle_logits = raw_middle_logits.max(dim=1).values
    elif pooling == "weighted":
        if layer_weights is None:
            weights = torch.ones(Lm, device=raw_middle_logits.device) / Lm
        else:
            weights = torch.tensor(layer_weights, device=raw_middle_logits.device, dtype=raw_middle_logits.dtype)
            weights = weights / weights.sum()
        middle_logits = (raw_middle_logits * weights.view(1, Lm, 1)).sum(dim=1)
    else:
        raise ValueError(f"Unsupported pooling: {pooling}")

    return middle_logits


def middle_alignment_loss(
    middle_feats,         # [B, Lm, D]
    class_text_feats,     # [C, D]
    labels,               # [B]
    logit_scale,
    layer_weights=None,
    pooling="weighted"
):
    raw_logits = logit_scale.exp() * torch.einsum("bld,cd->blc", middle_feats, class_text_feats)
    image_class_logits = aggregate_middle_logits(
        raw_logits,
        layer_weights=layer_weights,
        pooling=pooling
    )
    loss = F.cross_entropy(image_class_logits, labels)
    return loss, image_class_logits, raw_logits


# =========================================================
# 5. Text feature builders
# =========================================================
@torch.no_grad()
def build_class_text_features(clip_model, tokenizer, class_names, prompt_templates, device):
    prompts = [
        template.format(class_name)
        for class_name in class_names
        for template in prompt_templates
    ]
    tokens = tokenizer(prompts).to(device)

    text_features = clip_model.encode_text(tokens)
    text_features = text_features.view(len(class_names), len(prompt_templates), -1)
    text_features = text_features.mean(dim=1)   # [C, D]
    text_features = F.normalize(text_features, dim=-1)
    return text_features


# =========================================================
# 6. Optional future extension hooks
# =========================================================
def aggregate_text_groups(text_group_feats, group_weights=None, mode="weighted"):
    """
    Future extension hook.
    text_group_feats: [G, C, D] or list of [C, D]
    """
    if isinstance(text_group_feats, list):
        text_group_feats = torch.stack(text_group_feats, dim=0)

    G = text_group_feats.shape[0]

    if mode == "mean":
        return text_group_feats.mean(dim=0)
    elif mode == "weighted":
        if group_weights is None:
            weights = torch.ones(G, device=text_group_feats.device) / G
        else:
            weights = torch.tensor(group_weights, device=text_group_feats.device, dtype=text_group_feats.dtype)
            weights = weights / weights.sum()
        return (text_group_feats * weights.view(G, 1, 1)).sum(dim=0)
    else:
        raise ValueError(f"Unsupported mode: {mode}")


# =========================================================
# 7. Training
# =========================================================
def train_global_middle_dynamic(
    model,
    tokenizer,
    train_loader,
    class_names,
    prompt_templates,
    device,
    embed_dim=512,
    candidate_middle_layers=[2, 5, 8, 11],
    topk_middle=4,
    final_middle_layer=11,
    tau_middle=1.0,
    layer_weight_mode="softmax",
    use_middle=True,
    lambda_middle=0.1,
    lr=1e-3,
    num_epochs=30,
    middle_pooling="weighted",
):

    # -------- dynamic layer preparation --------
    layer_config = prepare_dynamic_layer_config(
        model=model,
        train_loader=train_loader,
        device=device,
        candidate_middle_layers=candidate_middle_layers,
        topk_middle=topk_middle,
        final_middle_layer=final_middle_layer,
        tau_middle=tau_middle,
        weight_mode=layer_weight_mode,
    )

    selected_middle_layers = layer_config["selected_middle_layers"]
    middle_weights = layer_config["middle_weights"]

    print("Selected middle layers:", selected_middle_layers)
    print("Middle layer weights:", [round(w, 4) for w in middle_weights])

    adapter_model = CLIP_DA_GlobalMiddle(
        embed_dim=embed_dim,
        clip_model=model,
        middle_layers=selected_middle_layers,
        use_middle=use_middle,
    ).to(device)

    optimizer_model = torch.optim.AdamW(
        adapter_model.parameters(),
        lr=lr,
        weight_decay=1e-4,
        eps=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_model,
        num_epochs * len(train_loader)
    )

    with torch.no_grad():
        cached_class_text_feats = build_class_text_features(
            clip_model=model,
            tokenizer=tokenizer,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device
        )

    best_state = None
    best_train_total = float("inf")

    for epoch in range(num_epochs):
        adapter_model.train()

        running_global_loss = 0.0
        running_middle_loss = 0.0
        running_total_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.size(0)

            # instance-level text features for global branch
            prompts_instance = [
                template.format(class_names[l.item()])
                for l in labels
                for template in prompt_templates
            ]
            tokens_instance = tokenizer(prompts_instance).to(device)

            with torch.no_grad():
                text_feats_instance = model.encode_text(tokens_instance)
                text_feats_instance = text_feats_instance.view(batch_size, len(prompt_templates), -1)
                text_feats_instance = text_feats_instance.mean(dim=1)   # [B, D]
                text_feats_instance = F.normalize(text_feats_instance, dim=-1)

            global_img_feats, adapted_instance_txt_feats, middle_feats = adapter_model(
                images, text_feats_instance
            )

            # global loss
            loss_global = contrastive_loss(
                global_img_feats,
                adapted_instance_txt_feats,
                adapter_model.logit_scale
            )

            # class text feats for middle branch
            adapted_class_txt_feats = adapter_model.adapt_text_features(cached_class_text_feats)

            # middle loss
            if use_middle and middle_feats is not None:
                loss_middle, _, _ = middle_alignment_loss(
                    middle_feats=middle_feats,
                    class_text_feats=adapted_class_txt_feats,
                    labels=labels,
                    logit_scale=adapter_model.logit_scale,
                    layer_weights=middle_weights,
                    pooling=middle_pooling
                )
            else:
                loss_middle = torch.tensor(0.0, device=device)

            total_loss = loss_global + lambda_middle * loss_middle

            optimizer_model.zero_grad()
            total_loss.backward()
            optimizer_model.step()
            scheduler.step()

            running_global_loss += loss_global.item() * batch_size
            running_middle_loss += loss_middle.item() * batch_size
            running_total_loss += total_loss.item() * batch_size
            total_samples += batch_size

        avg_global = running_global_loss / total_samples
        avg_middle = running_middle_loss / total_samples
        avg_total = running_total_loss / total_samples

        if avg_total < best_train_total:
            best_train_total = avg_total
            best_state = {k: v.detach().cpu().clone() for k, v in adapter_model.state_dict().items()}

        print(
            f"Epoch {epoch+1:02d} | "
            f"global_loss: {avg_global:.4f} | "
            f"middle_loss: {avg_middle:.4f} | "
            f"total_loss: {avg_total:.4f} | "
            f"logit_scale: {adapter_model.logit_scale.exp().item():.4f}"
        )

    if best_state is not None:
        adapter_model.load_state_dict(best_state)

    train_meta = {
        "prompt_templates": prompt_templates,
        "layer_config": layer_config,
    }

    return adapter_model, train_meta


# =========================================================
# 8. Evaluation
# =========================================================
@torch.no_grad()
def evaluate_global_middle_dynamic(
    adapter_model,
    clip_model,
    data_loader,
    tokenizer,
    class_names,
    prompt_templates,
    device,
    middle_weights=None,
    beta_middle=0.2,
    use_middle=True,
    max_batch=None,
    middle_pooling="weighted",
    return_details=False
):
    adapter_model.eval()

    class_text_feats = build_class_text_features(
        clip_model=clip_model,
        tokenizer=tokenizer,
        class_names=class_names,
        prompt_templates=prompt_templates,
        device=device
    )

    all_labels = []
    all_global_logits = []
    all_middle_logits = []
    all_fused_logits = []
    all_probs = []
    all_preds = []

    total_loss = 0.0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(data_loader):
        if max_batch is not None and batch_idx >= max_batch:
            break

        images = images.to(device)
        labels = labels.to(device)
        batch_size = images.size(0)

        global_img_feats, adapted_class_txt_feats, middle_feats = adapter_model(
            images, class_text_feats
        )

        scale = adapter_model.logit_scale.exp()
        global_logits = scale * (global_img_feats @ adapted_class_txt_feats.T)

        if use_middle and middle_feats is not None:
            _, middle_logits, _ = middle_alignment_loss(
                middle_feats=middle_feats,
                class_text_feats=adapted_class_txt_feats,
                labels=labels,
                logit_scale=adapter_model.logit_scale,
                layer_weights=middle_weights,
                pooling=middle_pooling
            )
        else:
            middle_logits = torch.zeros_like(global_logits)

        fused_logits = global_logits + beta_middle * middle_logits

        probs = F.softmax(fused_logits, dim=1)
        preds = probs.argmax(dim=1)
        loss = F.cross_entropy(fused_logits, labels)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.append(labels.cpu())
        all_global_logits.append(global_logits.cpu())
        all_middle_logits.append(middle_logits.cpu())
        all_fused_logits.append(fused_logits.cpu())
        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_global_logits = torch.cat(all_global_logits).numpy()
    all_middle_logits = torch.cat(all_middle_logits).numpy()
    all_fused_logits = torch.cat(all_fused_logits).numpy()
    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()

    results = {
        "loss": total_loss / max(total_samples, 1),
        "acc": (all_preds == all_labels).mean(),
    }

    if return_details:
        results.update({
            "labels": all_labels,
            "global_logits": all_global_logits,
            "middle_logits": all_middle_logits,
            "fused_logits": all_fused_logits,
            "probs": all_probs,
            "preds": all_preds,
        })

    return results


# =========================================================
# 9. Plotting
# =========================================================
def plot_eval_results(labels, probs, preds, class_names):
    acc = (preds == labels).mean()
    print(f"Accuracy: {acc:.4f}")

    cm = confusion_matrix(labels, preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    num_classes = len(class_names)
    labels_onehot = F.one_hot(torch.tensor(labels), num_classes=num_classes).numpy()

    try:
        aucs = roc_auc_score(labels_onehot, probs, average=None, multi_class="ovr")
        for i, auc in enumerate(aucs):
            print(f"Class {class_names[i]}: AUROC = {auc:.4f}")
        print(f"Averaged AUROC: {aucs.mean():.4f}")
    except Exception as e:
        print("Error when calculating AUROC:", e)

In [62]:
# prompt_templates = [
#         "a photo of chest X-ray showing {}",
#         "a chest X-ray image showing {}.",
#         # "evidence of {} in lungs",
#         # "radiographic signs of {}.",
#         # "a patient diagnosed with {} in chest X-ray."
#     ]
prompt_templates = [
        "a photo of Ultrasound showing {}",
        "an Ultrasound image showing {}.",
    ]
# prompt_templates = [
#         "a photo of CT showing {}",
#         "a CT image showing {}.",
#     ]
# prompt_templates = [
#         "a photo of histopathology showing {}",
#         "a histopathology image showing {}.",
#     ]

In [63]:
# template = 'this is a photo of chest X-ray showing'
# text_tokens = tokenizer([template + l for l in class_names]).to(device)
from dataset.dataset import Chestxray14_Dataset
import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" 
# Basic transform compatible with CLIP input size & normalization
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),           
    transforms.Lambda(lambda img: img.convert("RGB")),      
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711))  
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),       
    transforms.Lambda(lambda img: img.convert("RGB")),      
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711))  
])

# dataset_root = r"D:\MedicalVLMs\data\COVIDxCXR4"
# dataset_root = r'D:\MedicalVLMs\data\Chest X-Ray Images (Pneumonia)'
# dataset_root = r'D:\MedicalVLMs\data\CheXpert_5x1000'
# dataset_root = r"D:\MedicalVLMs\data\Tuberculosis Chest X-rays (Shenzhen)"
dataset_root = r"D:\MedicalVLMs\exp_datasets\ultrasound\BUSI"
# dataset_root = r'D:\MedicalVLMs\exp_datasets\BTMRI'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\CTKi'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\SCCT'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\LC25000'
fewshot_dataset = FewShotImageFolder(root=dataset_root, num_shots=16, transform=transform, test_transform = test_transform, seed=2024)

from torch.utils.data import DataLoader
train_loader = DataLoader(fewshot_dataset, batch_size=32, shuffle=True)

test_dataset = fewshot_dataset.get_test_set()
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(fewshot_dataset)}")
print(f"Test size: {len(test_dataset)}")


idx_to_folder = {v: k for k, v in fewshot_dataset.class_to_idx.items()}
class_names = [folder_name for idx, folder_name in sorted(idx_to_folder.items())]
print(class_names)

Train size: 48
Test size: 732
['benign tumor', 'malignant tumor', 'normal']


In [ ]:

import numpy as np
import torch

shot_list = [1, 2, 4, 8, 16]
seed_list = list(range(2000, 2050))   

results_all = {
    "shot": [],
    "fused_acc_mean": [],
    "fused_acc_std": [],
    "global_acc_mean": [],
    "global_acc_std": [],
    "middle_acc_mean": [],
    "middle_acc_std": [],
    "fused_auc_mean": [],
    "fused_auc_std": [],
}

for shot in shot_list:
    print("\n" + "="*60)
    print(f"Running {shot}-shot experiment (multi-seed)")
    print("="*60)

    fused_acc_list = []
    global_acc_list = []
    middle_acc_list = []
    fused_auc_list = []

    for seed in seed_list:
        print(f"\n--- Seed {seed} ---")

        # =========================
        # 1. Dataset
        # =========================
        fewshot_dataset = FewShotImageFolder(
            root=dataset_root,
            num_shots=shot,
            transform=transform,
            test_transform=test_transform,
            seed=seed
        )

        train_loader = DataLoader(fewshot_dataset, batch_size=32, shuffle=True)

        test_dataset = fewshot_dataset.get_test_set()
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

        idx_to_folder = {v: k for k, v in fewshot_dataset.class_to_idx.items()}
        class_names = [folder_name for idx, folder_name in sorted(idx_to_folder.items())]

        # =========================
        # 2. Train
        # =========================
        adapter_model, train_meta = train_global_middle_dynamic(
            model=model,
            tokenizer=tokenizer,
            train_loader=train_loader,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device,
            embed_dim=512,
            candidate_middle_layers=[5, 6, 7, 8, 9, 10, 11],
            topk_middle=4,
            final_middle_layer=11,
            tau_middle=1.0,
            layer_weight_mode="softmax",
            use_middle=True,
            lambda_middle=0.1,
            lr=1e-3,
            num_epochs=50,
            middle_pooling="weighted",
        )
        adapter_model.to(device)

        layer_config = train_meta["layer_config"]

        # =========================
        # 3. Eval
        # =========================
        test_metrics = evaluate_global_middle_dynamic(
            adapter_model=adapter_model,
            clip_model=model,
            data_loader=test_loader,
            tokenizer=tokenizer,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device,
            middle_weights=layer_config["middle_weights"],
            beta_middle=0.2,
            use_middle=True,
            return_details=True
        )
    #     test_metrics = evaluate_global_middle_dynamic(
    #     adapter_model=adapter_model,
    #     clip_model=model,
    #     data_loader=test_loader,
    #     tokenizer=tokenizer,
    #     class_names=class_names,
    #     prompt_templates=prompt_templates,
    #     device=device,
    #     middle_weights=layer_config["middle_weights"],
    #     use_middle=True,
    #     return_details=True
    # )

        # ===== fused =====
        fused_acc = test_metrics["acc"]

        # ===== global =====
        global_probs = torch.softmax(torch.tensor(test_metrics["global_logits"]), dim=1).numpy()
        global_preds = np.argmax(global_probs, axis=1)
        global_acc = (global_preds == test_metrics["labels"]).mean()

        # ===== middle =====
        middle_probs = torch.softmax(torch.tensor(test_metrics["middle_logits"]), dim=1).numpy()
        middle_preds = np.argmax(middle_probs, axis=1)
        middle_acc = (middle_preds == test_metrics["labels"]).mean()

        # ===== AUROC =====
        try:
            labels_onehot = torch.nn.functional.one_hot(
                torch.tensor(test_metrics["labels"]),
                num_classes=len(class_names)
            ).numpy()

            fused_auc = roc_auc_score(
                labels_onehot,
                test_metrics["probs"],
                average="macro",
                multi_class="ovr"
            )
        except:
            fused_auc = np.nan

        # collect
        fused_acc_list.append(fused_acc)
        global_acc_list.append(global_acc)
        middle_acc_list.append(middle_acc)
        fused_auc_list.append(fused_auc)

    # =========================
    # 4. Aggregate
    # =========================
    def mean_std(x):
        return np.mean(x), np.std(x)

    fused_mean, fused_std = mean_std(fused_acc_list)
    global_mean, global_std = mean_std(global_acc_list)
    middle_mean, middle_std = mean_std(middle_acc_list)
    auc_mean, auc_std = mean_std(fused_auc_list)

    results_all["shot"].append(shot)
    results_all["fused_acc_mean"].append(fused_mean)
    results_all["fused_acc_std"].append(fused_std)
    results_all["global_acc_mean"].append(global_mean)
    results_all["global_acc_std"].append(global_std)
    results_all["middle_acc_mean"].append(middle_mean)
    results_all["middle_acc_std"].append(middle_std)
    results_all["fused_auc_mean"].append(auc_mean)
    results_all["fused_auc_std"].append(auc_std)

    print("\n================ RESULT ================")
    print(f"{shot}-shot:")
    print(f"Fused  Acc: {fused_mean:.4f} ± {fused_std:.4f}")
    print(f"Global Acc: {global_mean:.4f} ± {global_std:.4f}")
    print(f"Middle Acc: {middle_mean:.4f} ± {middle_std:.4f}")
    print(f"Fused  AUC: {auc_mean:.4f} ± {auc_std:.4f}")


Running 1-shot experiment (multi-seed)

--- Seed 2000 ---
Selected middle layers: [6, 7, 8, 11]
Middle layer weights: [0.2418, 0.2429, 0.242, 0.2733]
Epoch 01 | global_loss: 1.6491 | middle_loss: 2.0597 | total_loss: 1.8550 | logit_scale: 85.1471
Epoch 02 | global_loss: 2.7081 | middle_loss: 3.4155 | total_loss: 3.0497 | logit_scale: 85.0661
Epoch 03 | global_loss: 1.0977 | middle_loss: 4.6343 | total_loss: 1.5611 | logit_scale: 84.9898
Epoch 04 | global_loss: 1.1730 | middle_loss: 2.9754 | total_loss: 1.4706 | logit_scale: 84.9160
Epoch 05 | global_loss: 0.8733 | middle_loss: 1.8881 | total_loss: 1.0621 | logit_scale: 84.8473
Epoch 06 | global_loss: 0.3248 | middle_loss: 1.0307 | total_loss: 0.4279 | logit_scale: 84.7917
Epoch 07 | global_loss: 0.2709 | middle_loss: 1.1253 | total_loss: 0.3835 | logit_scale: 84.7458
Epoch 08 | global_loss: 0.1764 | middle_loss: 1.2646 | total_loss: 0.3028 | logit_scale: 84.7080
Epoch 09 | global_loss: 0.3227 | middle_loss: 1.1380 | total_loss: 0.4365